<a href="https://colab.research.google.com/github/rakesh-mandal/ML/blob/main/day40-Interative%20Imputer/MICE_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [5]:
df = pd.DataFrame({
    'Experience_Years': [2.0, np.nan, 12.0, 7.0, 15.0, 1.0],
    'Education': ['Bachelors', 'Masters', 'Masters', np.nan, 'Masters', 'High School'],
    'City': ['Delhi', 'New York', np.nan, 'Tokyo', 'Delhi', 'Tokyo'],
    'Promoted': [0, 1, 1, 0, 1, 0] # Our target variable (y)
})

In [6]:
df

,Experience_Years,Education,City,Promoted
0,2.0,Bachelors,Delhi,0
1,NaN,Masters,New York,1
2,12.0,Masters,NaN,1
3,7.0,NaN,Tokyo,0
4,15.0,Masters,Delhi,1
5,1.0,High School,Tokyo,0


In [7]:
X = df.drop(columns=['Promoted'])
y = df['Promoted']

In [8]:
# --- 2. Setup Individual Categorical Encoders ---

In [9]:
# Ordinal Encoding setup (mapping strings to explicit numeric steps, keeping NaNs as NaN)
edu_order = [['High School', 'Bachelors', 'Masters']]
ordinal_processor = OrdinalEncoder(
    categories=edu_order,
    handle_unknown='use_encoded_value',
    unknown_value=np.nan,
    encoded_missing_value=np.nan
)

In [10]:
# Nominal One-Hot setup (ignoring unknowns keeps rows with missing structural elements safely as numeric NaNs)
nominal_processor = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', ['Experience_Years']),
        ('ord', ordinal_processor, ['Education']),
        ('nom', nominal_processor, ['City'])
    ],
    remainder='drop'
)

In [12]:
pipeline = Pipeline(steps=[
    ('encoding_layer', preprocessor),
    ('imputation_layer', IterativeImputer(max_iter=10, random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [13]:
# --- 5. Train and Predict using the Pipeline ---
# The pipeline handles the encoding transform, tracks data columns natively,
# imputes numerical properties seamlessly, and trains the random forest classifier.
pipeline.fit(X, y)

Pipeline(steps=[('encoding_layer',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Experience_Years']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['High '
                                                                              'School',
                                                                              'Bachelors',
                                                                              'Masters']],
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=nan),
                                                  ['Education']),
                                                 ('nom',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['City'])])),
                ('imputation_layer', IterativeImputer(random_state=42)),
                ('classifier', RandomForestClassifier(random_state=42))])

In [14]:
print("Pipeline compiled and trained successfully!")
print("Predictions on the training set:", pipeline.predict(X))

Pipeline compiled and trained successfully!
Predictions on the training set: [0 1 1 0 1 0]


In [15]:
from sklearn.metrics import accuracy_score

In [16]:
y_pred=pipeline.predict(X)

In [17]:
y_pred

array([0, 1, 1, 0, 1, 0])

In [18]:
accuracy_score(y_pred, y)

1.0

In [19]:
# =====================================================================
# --- NEW STEP: EXTRACT AND DISPLAY THE IMIPUTED DATAFRAME ---
# =====================================================================

In [21]:
# 1. Slice the pipeline to include only the encoding and imputation steps
imputation_pipeline = Pipeline(steps=pipeline.steps[:-1])
imputation_pipeline

Pipeline(steps=[('encoding_layer',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Experience_Years']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['High '
                                                                              'School',
                                                                              'Bachelors',
                                                                              'Masters']],
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=nan),
                                                  ['Education']),
                                                 ('nom',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['City'])])),
                ('imputation_layer', IterativeImputer(random_state=42))])

In [23]:
# 2. Transform the raw data through these layers
filled_array = imputation_pipeline.transform(X)
filled_array

array([[ 2.        ,  1.        ,  1.        ,  0.        ,  0.        ,
         0.        ],
       [11.69118688,  2.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [12.        ,  2.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 7.        ,  1.17882922,  0.        ,  0.        ,  1.        ,
         0.        ],
       [15.        ,  2.        ,  1.        ,  0.        ,  0.        ,
         0.        ],
       [ 1.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ]])

In [25]:
# 3. Reconstruct feature names to make the output readable
# Get feature names from the encoding layer
encoded_feature_names = pipeline.named_steps['encoding_layer'].get_feature_names_out()
encoded_feature_names

array(['num__Experience_Years', 'ord__Education', 'nom__City_Delhi',
       'nom__City_New York', 'nom__City_Tokyo', 'nom__City_nan'],
      dtype=object)

In [26]:
# Create a clean DataFrame
df_filled = pd.DataFrame(filled_array, columns=encoded_feature_names)

In [27]:
print("=== The Completely Filled (Imputed) Numeric DataFrame ===")
print(np.round(df_filled, 2))

=== The Completely Filled (Imputed) Numeric DataFrame ===
   num__Experience_Years  ord__Education  nom__City_Delhi  nom__City_New York  \
0                   2.00            1.00              1.0                 0.0   
1                  11.69            2.00              0.0                 1.0   
2                  12.00            2.00              0.0                 0.0   
3                   7.00            1.18              0.0                 0.0   
4                  15.00            2.00              1.0                 0.0   
5                   1.00            0.00              0.0                 0.0   

   nom__City_Tokyo  nom__City_nan  
0              0.0            0.0  
1              0.0            0.0  
2              0.0            1.0  
3              1.0            0.0  
4              0.0            0.0  
5              1.0            0.0  


In [28]:
df_filled

,num__Experience_Years,ord__Education,nom__City_Delhi,nom__City_New York,nom__City_Tokyo,nom__City_nan
0,2.000000,1.000000,1.0,0.0,0.0,0.0
1,11.691187,2.000000,0.0,1.0,0.0,0.0
2,12.000000,2.000000,0.0,0.0,0.0,1.0
3,7.000000,1.178829,0.0,0.0,1.0,0.0
4,15.000000,2.000000,1.0,0.0,0.0,0.0
5,1.000000,0.000000,0.0,0.0,1.0,0.0
